In [11]:

import gymnasium as gym
import torch
from pathlib import Path
from torch.distributions import Categorical
import torch.nn as nn

DEVICE = torch.device("cuda")


class ActorCriticNet(nn.Module):
    def __init__(self, obs_dim=8, n_actions=4):
        super().__init__()

        self.actor = nn.Sequential(
          nn.Linear(obs_dim, 128),
          nn.LayerNorm(128),
          nn.Tanh(),
          nn.Linear(128, 256),
          nn.LayerNorm(256),
          nn.Tanh(),
          nn.Linear(256, n_actions)
        )

        self.critic = nn.Sequential(
          nn.Linear(obs_dim, 128),
          nn.LayerNorm(128),
          nn.Tanh(),
          nn.Linear(128, 256),
          nn.LayerNorm(256),
          nn.Tanh(),
          nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.actor(x), self.critic(x).squeeze(-1)

# 1. 저장된 모델 불러오기
model_path = Path("./ac_lunar_results/best_model-1.pth")

checkpoint = torch.load(
    model_path,
    map_location=DEVICE,
    weights_only=True
)

ac = ActorCriticNet(8, 4).to(DEVICE)
ac.load_state_dict(checkpoint["state_dict"])
ac.eval()

# 2. 게임 화면 생성
for seed in (42*i for i in range(1, 42)):
  print(f"seed {seed} test")
  env = gym.make("LunarLander-v3", render_mode="human")

  obs, info = env.reset(seed=seed)
  total_reward = 0

  # 3. 학습된 Actor로 게임 실행
  with torch.no_grad():
      for t in range(1500):

          state = torch.as_tensor(
              obs,
              dtype=torch.float32,
              device=DEVICE
          )

          logits, value = ac(state)

          # 가장 확률 높은 행동 선택
          # action = torch.argmax(logits, dim=-1).item()

          # 샘플링하기
          dist = Categorical(logits=logits)
          action = dist.sample().item()

          obs, reward, terminated, truncated, info = env.step(action)

          total_reward += float(reward)

          if terminated or truncated:
              break

  env.close()

  print(f"Total Reward: {total_reward:.2f}")
  print(f"Total Steps: {t + 1}")


seed 42 test
Total Reward: 230.98
Total Steps: 435
seed 84 test
Total Reward: -121.22
Total Steps: 1000
seed 126 test
Total Reward: 264.27
Total Steps: 374
seed 168 test
Total Reward: 276.02
Total Steps: 351
seed 210 test
Total Reward: 2.15
Total Steps: 167
seed 252 test
Total Reward: -25.84
Total Steps: 184
seed 294 test
Total Reward: 278.62
Total Steps: 397
seed 336 test
Total Reward: 209.02
Total Steps: 517
seed 378 test
Total Reward: 12.33
Total Steps: 229
seed 420 test
Total Reward: 25.31
Total Steps: 237
seed 462 test
Total Reward: -153.55
Total Steps: 1000
seed 504 test


KeyboardInterrupt: 

In [ ]:
import torch

pth = torch.load(Path(r"C:\LANG_CHAIN_2026\2026-05-19_KDT_lang_chain\01_class\2026_10_08_multimodal\ac_lunar_results\tuned_online_ac"))
pth